# DSAA2011 Final Project - Student Dropout Dataset

This notebook provides the full polished pipeline using **pandas, numpy, matplotlib, seaborn, and scikit-learn**.
It covers:
- preprocessing
- visualization (including t-SNE)
- clustering
- multiclass prediction
- full confusion matrices and per-class performance metrics
- ROC-AUC (OvR)


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
os.makedirs('results_ml/figures', exist_ok=True)
os.makedirs('results_ml/tables', exist_ok=True)


In [ ]:
# Load dataset
raw = pd.read_csv('data.csv', sep=';', encoding='utf-8-sig')
raw.head()


In [ ]:
# Basic checks
print('Shape:', raw.shape)
print('Missing values:', raw.isna().sum().sum())
print(raw['Target'].value_counts())

# Save missing value report
raw.isna().sum().to_csv('results_ml/tables/missing_values.csv', header=['missing_count'])


In [ ]:
# Features / labels
X = raw.drop(columns=['Target'])
y = raw['Target']  # Keep multiclass labels: Dropout, Enrolled, Graduate

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# Standardized features for unsupervised analysis / visualization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 1) Visualization (PCA + t-SNE)


In [ ]:
# PCA 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({'PC1': X_pca[:,0], 'PC2': X_pca[:,1], 'Target': y.values})

plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Target', alpha=0.65, s=20)
plt.title('PCA 2D Projection by Target')
plt.tight_layout()
plt.savefig('results_ml/figures/pca_2d.png', dpi=200)
plt.show()


In [ ]:
# t-SNE 2D (run on full set; if slow, sample)
X_tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_STATE,
).fit_transform(X_scaled)

tsne_df = pd.DataFrame({'TSNE1': X_tsne[:,0], 'TSNE2': X_tsne[:,1], 'Target': y.values})

plt.figure(figsize=(9, 7))
sns.scatterplot(data=tsne_df, x='TSNE1', y='TSNE2', hue='Target', alpha=0.65, s=20)
plt.title('t-SNE 2D Projection by Target')
plt.tight_layout()
plt.savefig('results_ml/figures/tsne_2d.png', dpi=200)
plt.show()

# save embedding points
pd.concat([tsne_df, pca_df[['PC1','PC2']]], axis=1).to_csv('results_ml/tables/embedding_points.csv', index=False)


## 2) Clustering Analysis


In [ ]:
cluster_results = []

# KMeans
km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=20)
km_labels = km.fit_predict(X_scaled)
cluster_results.append([
    'KMeans',
    silhouette_score(X_scaled, km_labels),
    davies_bouldin_score(X_scaled, km_labels),
    calinski_harabasz_score(X_scaled, km_labels),
])

# Agglomerative
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X_scaled)
cluster_results.append([
    'Agglomerative',
    silhouette_score(X_scaled, agg_labels),
    davies_bouldin_score(X_scaled, agg_labels),
    calinski_harabasz_score(X_scaled, agg_labels),
])

cluster_df = pd.DataFrame(cluster_results, columns=['algorithm', 'silhouette', 'davies_bouldin', 'calinski_harabasz'])
cluster_df.to_csv('results_ml/tables/clustering_metrics.csv', index=False)
cluster_df


In [ ]:
# Visualize KMeans clusters on t-SNE space
tsne_cluster_df = tsne_df.copy()
tsne_cluster_df['kmeans_cluster'] = km_labels

plt.figure(figsize=(9, 7))
sns.scatterplot(data=tsne_cluster_df, x='TSNE1', y='TSNE2', hue='kmeans_cluster', palette='tab10', alpha=0.65, s=20)
plt.title('KMeans Clusters on t-SNE Space')
plt.tight_layout()
plt.savefig('results_ml/figures/kmeans_tsne_clusters.png', dpi=200)
plt.show()


## 3) Prediction Models (Multiclass)


In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE))
    ]),
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced'))
    ]),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model

print('Models trained:', list(fitted.keys()))


In [ ]:
# Evaluation helper with full confusion matrices and class-wise performance
labels = sorted(y.unique())

rows = []
for name, model in fitted.items():
    for split_name, X_part, y_part in [
        ('train', X_train, y_train),
        ('test', X_test, y_test),
        ('all', X, y),
    ]:
        y_pred = model.predict(X_part)
        acc = accuracy_score(y_part, y_pred)
        p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_part, y_pred, average='macro')
        p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(y_part, y_pred, average='weighted')

        rows.append([name, split_name, acc, p_macro, r_macro, f1_macro, p_weight, r_weight, f1_weight])

        cm = confusion_matrix(y_part, y_pred, labels=labels)
        cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels], columns=[f'pred_{l}' for l in labels])
        cm_df.to_csv(f'results_ml/tables/confusion_matrix_{name}_{split_name}.csv')

        plt.figure(figsize=(6, 5))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'{name} Confusion Matrix ({split_name})')
        plt.tight_layout()
        plt.savefig(f'results_ml/figures/confusion_matrix_{name}_{split_name}.png', dpi=200)
        plt.show()

        report = classification_report(y_part, y_pred, output_dict=True)
        pd.DataFrame(report).T.to_csv(f'results_ml/tables/classification_report_{name}_{split_name}.csv')

perf_df = pd.DataFrame(rows, columns=['model','split','accuracy','precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted'])
perf_df.to_csv('results_ml/tables/model_performance.csv', index=False)
perf_df


In [ ]:
# Multiclass ROC-AUC (One-vs-Rest)
y_binarized = label_binarize(y, classes=labels)

for name, model in fitted.items():
    proba = model.predict_proba(X)
    plt.figure(figsize=(7, 6))
    auc_rows = []
    for i, cls in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_binarized[:, i], proba[:, i])
        roc_auc = auc(fpr, tpr)
        auc_rows.append([cls, roc_auc])
        plt.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')

    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{name} ROC Curves (OvR)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'results_ml/figures/roc_{name}.png', dpi=200)
    plt.show()

    pd.DataFrame(auc_rows, columns=['class','auc']).to_csv(f'results_ml/tables/auc_{name}.csv', index=False)


## 4) Hyperparameter Validation (GridSearchCV)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Logistic Regression tuning
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE))
])

lr_grid = {
    'clf__C': [0.1, 0.5, 1.0, 2.0],
    'clf__penalty': ['l2'],
}

lr_search = GridSearchCV(lr_pipe, lr_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
lr_search.fit(X_train, y_train)
print('Best LR:', lr_search.best_params_, 'Score:', lr_search.best_score_)

# Random Forest tuning
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
])

rf_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [None, 12, 20],
    'clf__min_samples_split': [2, 6],
}

rf_search = GridSearchCV(rf_pipe, rf_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
rf_search.fit(X_train, y_train)
print('Best RF:', rf_search.best_params_, 'Score:', rf_search.best_score_)

pd.DataFrame([
    ['LogisticRegression', str(lr_search.best_params_), lr_search.best_score_],
    ['RandomForest', str(rf_search.best_params_), rf_search.best_score_],
], columns=['model','best_params','best_cv_f1_weighted']).to_csv('results_ml/tables/validation_summary.csv', index=False)


## Final notes

- All major requirement items are implemented in this notebook.
- Exported tables and figures are saved under `results_ml/`.
- This notebook is ready for course submission after execution in a Python environment with dependencies installed.
